In [97]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.multioutput import MultiOutputClassifier

from data_utils import extract_contracts, create_labeled_chunks, multihot_encode

import re
import json
import numpy as np

Data Preparation (see notebooks 4-5)

In [98]:
with open(r"..\data\train.json") as file:
    train_data = json.load(file)

with open(r"..\data\validation.json") as file:
    val_data = json.load(file)

with open(r"..\data\test.json") as file:
    test_data = json.load(file)


len(train_data['data']), len(val_data['data']), len(test_data['data'])

(326, 82, 102)

In [99]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    use_fast=True
)

In [100]:
train_contracts = extract_contracts(train_data)
val_contracts = extract_contracts(val_data)
test_contracts = extract_contracts(test_data)

train_chunks = [
    chunk 
    for contract in train_contracts 
    for chunk in create_labeled_chunks(contract, tokenizer)
]

val_chunks = [
    chunk
    for contract in val_contracts
    for chunk in create_labeled_chunks(contract, tokenizer) 
]

test_chunks = [
    chunk
    for contract in test_contracts
    for chunk in create_labeled_chunks(contract, tokenizer) 
]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (11248 > 512). Running this sequence through the model will result in indexing errors


In [101]:
label_vocab = sorted({
    re.findall(r'"([^"]+)"', qa['question'])[0]
    for contract in train_contracts
    for qa in contract['qas']
})

label_to_id = {label: i for i, label in enumerate(label_vocab)}
id_to_label = {i: label for i, label in enumerate(label_vocab)}

train_labels = np.array([multihot_encode(chunk['labels'], label_to_id) for chunk in train_chunks])
val_labels = np.array([multihot_encode(chunk['labels'], label_to_id) for chunk in val_chunks])
test_labels = np.array([multihot_encode(chunk['labels'], label_to_id) for chunk in test_chunks])

```



```

Initiailze a TF-IDF vectorizer and fit it to our train_texts. Use the learned IDF scores (via transform method) to transform the rest of the training sets.

In [102]:
train_texts = np.array([chunk["text"] for chunk in train_chunks])
val_texts = np.array([chunk["text"] for chunk in val_chunks])
test_texts = np.array([chunk["text"] for chunk in test_chunks])

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_val_tfidf = vectorizer.transform(val_texts)
X_test_tfidf = vectorizer.transform(test_texts)

X_train_tfidf.shape

(35200, 26930)

Fit a LogisticRegression classifier, using the TF-IDF score as inputs. Note: MultiOutputClassifier trains a seperate LogistricRegression classifier for each clause label.

In [104]:
classifier = MultiOutputClassifier(LogisticRegression())
classifier.fit(X_train_tfidf, train_labels)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,LogisticRegression()
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Class labels.",list,"[array([0, 1]), array([0, 1]), array([0, 1]), array([0, 1]), ...]"
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[LogisticRegression(), LogisticRegression(), LogisticRegression(), LogisticRegression(), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,26930
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001


In [105]:
predictions = classifier.predict(X_val_tfidf)

accuracy = accuracy_score(val_labels, predictions, )
print("Accuracy: ", round(accuracy, 3))

Accuracy:  0.793


In [106]:
import pandas as pd

y_true = np.asarray(val_labels)
y_pred = np.asarray(predictions)

rows = []
for i, label in enumerate(label_vocab):
    rows.append({
        "label": label,
        "accuracy": accuracy_score(y_true[:, i], y_pred[:, i]),
        "negative_pct": (y_true[:, i] == 0).sum() / len(y_true),
        "micro_f1": np.nan,
        "macro_f1": np.nan,
        "positive_count": int(y_true[:, i].sum()),
        "negative_count": int((y_true[:, i] == 0).sum()),
    })

rows.append({
    "label": "OVERALL",
    "accuracy": np.nan,
    "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "positive_count": int(y_true.sum()),
    "negative_count": int((y_true == 0).sum()),
})

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(3))

,label,accuracy,negative_pct,micro_f1,macro_f1,positive_count,negative_count
0,Affiliate License-Licensee,0.998,0.998,NaN,NaN,15,7024
1,Affiliate License-Licensor,0.998,0.998,NaN,NaN,11,7028
2,Agreement Date,0.993,0.988,NaN,NaN,83,6956
3,Anti-Assignment,0.988,0.981,NaN,NaN,137,6902
4,Audit Rights,0.991,0.987,NaN,NaN,93,6946
5,Cap On Liability,0.986,0.979,NaN,NaN,146,6893
6,Change Of Control,0.992,0.992,NaN,NaN,57,6982
7,Competitive Restriction Exception,0.996,0.996,NaN,NaN,29,7010
8,Covenant Not To Sue,0.993,0.993,NaN,NaN,49,6990
9,Document Name,0.992,0.987,NaN,NaN,88,6951


In [107]:
metrics_df['negative_pct'].mean()

np.float64(0.9912958811361092)

Conclusion:

The baseline TF-IDF Logistic Regression Classifier has an overall accuracy of 79.3%.

However, this number is not a true representative of the predictive power of this model.

Analyzing the negative vs positive labels quickly reveals that each clause label is **heavily** imbalanced, with an average of **99.1%** of the labels being negative.

The F1 score of (0.398 micro, 0.238 macro) better explains the performance of this model, as it takes into account the imbalance of the dataset.